ESPERA veo muchos datos nulos porque hay variables que estás incluyendo que ya habia descartado previamente, la idea es incluir exclusivamente estas variables en los csv de DATASET TEST y DATASET TRAIN:


SEXO, ES_PUEBLO_ORIGINARIO, ES_EXTRANJERO, PREVISION, SERVICIO_SALUD, TIPO_PROCEDENCIA, ESPECIALIDAD_MEDICA, TIPO_ACTIVIDAD, SERVICIOINGRESO, TRASLADO_MACROZONA, TIPO_INGRESO, EDAD, CARGA_ONCOLOGICA, NUM_COMORBILIDADES, COMORBILIDAD_PRINCIPAL, MORTALIDAD, SEVERIDAD, CONSUMO_RECURSOS

In [34]:
import pandas as pd
import os

# 1. Definición de Rutas
carpeta = "../../Datos/Datos oncológicos (procesados)"
carpeta_salida = "../../Datos/Datos oncológicos de entrenamiento y prueba (ML)"
os.makedirs(carpeta_salida, exist_ok=True)

archivos = [
    "GRD_ONCOLOGIA_PROC_2019.csv",
    "GRD_ONCOLOGIA_PROC_2020.csv",
    "GRD_ONCOLOGIA_PROC_2021.csv",
    "GRD_ONCOLOGIA_PROC_2022.csv",
    "GRD_ONCOLOGIA_PROC_2023.csv",
    "GRD_ONCOLOGIA_PROC_2024.csv"
]

# Las 19 variables de oro definitivas
columnas_finales = [
    "SEXO", "ES_PUEBLO_ORIGINARIO", "ES_EXTRANJERO", "PREVISION", 
    "SERVICIO_SALUD", "TIPO_PROCEDENCIA", "ESPECIALIDAD_MEDICA", 
    "TIPO_ACTIVIDAD", "SERVICIOINGRESO", "TRASLADO_MACROZONA", 
    "TIPO_INGRESO", "EDAD", "CARGA_ONCOLOGICA", "NUM_COMORBILIDADES", 
    "COMORBILIDAD_PRINCIPAL", "CATEGORIA_CANCER", 
    "MORTALIDAD", "SEVERIDAD", "CONSUMO_RECURSOS"
]

# Variables de la lista anterior a las que aplicaremos One-Hot Encoding
cols_categoricas = [
    "SEXO", 
    "PREVISION", 
    "SERVICIO_SALUD", 
    "TIPO_PROCEDENCIA", 
    "ESPECIALIDAD_MEDICA", 
    "TIPO_ACTIVIDAD", 
    "SERVICIOINGRESO", 
    "TIPO_INGRESO", 
    "COMORBILIDAD_PRINCIPAL",
    "CATEGORIA_CANCER" # Agregada a la codificación
]

print("1. Extrayendo variables seleccionadas y uniendo datasets...")
lista_dfs = []

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # Extraemos solo las 19 columnas que nos interesan
    df_filtrado = df[columnas_finales].copy()
    
    # Creamos la columna temporal para recordar el año (para el split final)
    df_filtrado["ANIO_COHORTE"] = int(archivo[-8:-4]) 
    
    lista_dfs.append(df_filtrado)

df_completo = pd.concat(lista_dfs, ignore_index=True)

print("2. Aplicando One-Hot Encoding global...")
# Se aplica el OHE solo a las categóricas; las numéricas y binarias quedan intactas
df_encoded = pd.get_dummies(df_completo, columns=cols_categoricas, drop_first=False, dtype=int)

# Limpiar nombres de columnas (Evita errores de sintaxis en XGBoost)
df_encoded.columns = df_encoded.columns.str.replace(r'[<,>,\[,\],\s]+', '_', regex=True)
df_encoded.columns = df_encoded.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True)
df_encoded.columns = df_encoded.columns.str.upper()

print("3. Separando en Train (2019-2023) y Test (2024)...")
# Separación usando la columna temporal
df_train = df_encoded[df_encoded["ANIO_COHORTE"] <= 2023].copy()
df_test = df_encoded[df_encoded["ANIO_COHORTE"] == 2024].copy()

# Borramos la columna temporal para que no estorbe en el modelo
df_train = df_train.drop(columns=["ANIO_COHORTE"])
df_test = df_test.drop(columns=["ANIO_COHORTE"])

# 4. Guardar los datasets finales
ruta_train = os.path.join(carpeta_salida, "DATASET_TRAIN_2019_2023.csv")
ruta_test = os.path.join(carpeta_salida, "DATASET_TEST_2024.csv")

df_train.to_csv(ruta_train, index=False)
df_test.to_csv(ruta_test, index=False)

print(f"Train (2019-2023) guardado con {df_train.shape[0]} filas y {df_train.shape[1]} columnas.")
print(f"Test (2024) guardado con {df_test.shape[0]} filas y {df_test.shape[1]} columnas.")

1. Extrayendo variables seleccionadas y uniendo datasets...
2. Aplicando One-Hot Encoding global...
3. Separando en Train (2019-2023) y Test (2024)...
Train (2019-2023) guardado con 215409 filas y 76 columnas.
Test (2024) guardado con 54573 filas y 76 columnas.


Auditoría Estadística Final del Espacio de Características


Tras la consolidación del conjunto de datos y la aplicación de la codificación de variables categóricas (One-Hot Encoding), se ejecutó una auditoría estadística sobre el conjunto de entrenamiento (2019-2023) para garantizar la estabilidad matemática del espacio de características y descartar sesgos predictivos.

1. Análisis de Varianza y RepresentatividadSe analizó la presencia de las 76 características generadas para identificar columnas con varianza cercana a cero (Near-Zero Variance). Se detectaron variables con una representatividad inferior al 1%, correspondientes a categorías clínicas específicas (ej. ESPECIALIDAD_MEDICA_PEDIATRIA con 0.82% y CATEGORIA_CANCER_C40-C41 (HUESO) con 0.84%). Dado que en oncología estas prevalencias representan cohortes clínicamente relevantes (aproximadamente 1.700 pacientes), se decidió conservarlas, apoyándose en la robustez de los algoritmos basados en ensambles de árboles (Random Forest y XGBoost) para manejar divisiones minoritarias sin incurrir en sobreajuste. Las categorías de registros administrativos nulos (PREVISION_NO_CONSIGNADO y PREVISION_NO_IDENTIFICADA), con prevalencias marginales (< 0.05%), fueron ignoradas por su nulo aporte de ganancia de información.

2. Prevención de Fuga de Información (Data Leakage)Para asegurar la viabilidad prospectiva del modelo, se evaluó la correlación monotónica de Spearman entre todos los predictores y las tres variables objetivo. De acuerdo con el protocolo metodológico, se estableció un umbral de exclusión para correlaciones superiores a 0.80.Los resultados confirmaron la ausencia de fuga de información (Target Leakage), identificando señales clínicas coherentes sin predictores anómalos:Mortalidad: Presentó sus mayores correlaciones positivas con los ingresos a MEDICINA_INTERNA_Y_CRITICA ($r_s = 0.285$) y admisiones NO_PROGRAMADAS / URGENCIA ($r_s = 0.278$).Severidad: Reflejó una fuerte correlación clínica estructural con la variable derivada NUM_COMORBILIDADES ($r_s = 0.614$) y la TIPO_ACTIVIDAD_HOSPITALIZACION ($r_s = 0.603$).Consumo de Recursos: Estuvo traccionado principalmente por el NUM_COMORBILIDADES ($r_s = 0.392$) y el tipo de actividad.Dado que el predictor más fuerte alcanzó una correlación máxima de 0.61, se descarta matemáticamente la presencia de variables "oráculo" con capacidad predictiva aislada extrema. Por consiguiente, se omite el filtro secundario de validación mediante árboles de decisión univariados (cuyo umbral de exclusión estaba fijado en un AUC-ROC > 0.90), validando definitivamente el conjunto de predictores.

3. Detección de MulticolinealidadEl escaneo de correlación entre características independientes arrojó tres pares con multicolinealidad perfecta ($r_s = -1.0$): Sexo (Hombre/Mujer), Tipo de Actividad (Ambulatoria/Hospitalización) y Tipo de Ingreso (Programado/No Programado). Este fenómeno responde a un artefacto matemático natural del One-Hot Encoding sobre variables originalmente binarias (la trampa de la variable ficticia). Dado que la estrategia de modelado principal recae en XGBoost y Random Forest, los cuales son inherentemente inmunes a la multicolinealidad al evaluar características de forma independiente en cada nodo, se decidió mantener la estructura completa para maximizar la interpretabilidad posterior mediante valores SHAP.

In [36]:
# 1. Definición de rutas
ruta_train = "../../Datos/Datos oncológicos de entrenamiento y prueba (ML)/DATASET_TRAIN_2019_2023.csv"
ruta_directorio = "../../Resultados/Resultados (etapa 1 y 2)/Análisis final"
os.makedirs(ruta_directorio, exist_ok=True)
ruta_archivo_txt = os.path.join(ruta_directorio, "analisis_estadistico_final.txt")

# Cargar el dataset de entrenamiento
df_train = pd.read_csv(ruta_train)

with open(ruta_archivo_txt, "w", encoding="utf-8") as f:
    f.write("============================================================\n")
    f.write("AUDITORÍA ESTADÍSTICA FINAL DEL DATASET DE ENTRENAMIENTO\n")
    f.write("============================================================\n\n")

    # --- 1. ANÁLISIS DE VARIANZA (PROPORCIÓN DE 1s) ---
    f.write("1. ANÁLISIS DE VARIANZA (Presencia de categorías OHE)\n")
    f.write("-" * 60 + "\n")
    
    # Calculamos la media (que en binario es el % de presencia)
    presencia = (df_train.mean() * 100).sort_values(ascending=False)
    
    f.write(f"Total de variables analizadas: {len(presencia)}\n\n")
    f.write("Distribución de presencia (Top 10 más frecuentes):\n")
    f.write(presencia.head(10).to_string() + "\n\n")
    
    # Detectar variables con baja varianza (< 1%)
    variables_raras = presencia[presencia < 1.0]
    if len(variables_raras) > 0:
        f.write(f"ALERTA: Se detectaron {len(variables_raras)} variables con presencia < 1%:\n")
        f.write(variables_raras.to_string() + "\n")
    else:
        f.write("No se detectaron variables con varianza extremadamente baja (< 1%).\n")
    
    f.write("\n" + "="*60 + "\n")
    
    # --- 2. ANÁLISIS DE CORRELACIÓN SPEARMAN (TARGET LEAKAGE) ---
    f.write("2. MATRIZ DE CORRELACIÓN DE SPEARMAN (Enfoque en Targets)\n")
    f.write("-" * 60 + "\n")
    
    corr_matrix = df_train.corr(method='spearman')
    targets = ["MORTALIDAD", "SEVERIDAD", "CONSUMO_RECURSOS"]
    
    for target in targets:
        f.write(f"\n>>> Análisis para el Objetivo: {target}\n")
        # Correlaciones con el target (excluyendo los otros targets)
        correlaciones = corr_matrix[target].drop(targets).sort_values(ascending=False)
        
        f.write("Top 10 Correlaciones Positivas (Aumentan el riesgo/valor):\n")
        f.write(correlaciones.head(10).to_string() + "\n")
        
        f.write("\nTop 10 Correlaciones Negativas (Disminuyen el riesgo/valor):\n")
        f.write(correlaciones.tail(10).to_string() + "\n")
        
        # Alerta de Leakage
        leakage = correlaciones[abs(correlaciones) > 0.80]
        if len(leakage) > 0:
            f.write(f"\n¡PELIGRO! Posible Data Leakage detectado para {target}:\n")
            f.write(leakage.to_string() + "\n")

    f.write("\n" + "="*60 + "\n")
    
    # --- 3. DETECCIÓN DE MULTICOLINEALIDAD ENTRE PREDICTORES ---
    f.write("3. DETECCIÓN DE MULTICOLINEALIDAD (Redundancia)\n")
    f.write("-" * 60 + "\n")
    
    pares_redundantes = []
    # Solo comparamos predictores entre sí
    predictores = [c for c in corr_matrix.columns if c not in targets]
    
    for i in range(len(predictores)):
        for j in range(i + 1, len(predictores)):
            c1 = predictores[i]
            c2 = predictores[j]
            coef = corr_matrix.loc[c1, c2]
            if abs(coef) > 0.85:
                pares_redundantes.append((c1, c2, coef))
                
    if len(pares_redundantes) > 0:
        f.write(f"Se detectaron {len(pares_redundantes)} pares con alta correlación (> 0.85):\n")
        for p1, p2, v in pares_redundantes:
            f.write(f" - {p1} <--> {p2} | Coef: {v:.4f}\n")
    else:
        f.write("No se detectó multicolinealidad crítica entre predictores.\n")

print(f"✔️ Análisis completo guardado en: {ruta_archivo_txt}")

✔️ Análisis completo guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis final\analisis_estadistico_final.txt


Tabla correlaciones

In [37]:
# 1. Definición de rutas
ruta_train = "../../Datos/Datos oncológicos de entrenamiento y prueba (ML)/DATASET_TRAIN_2019_2023.csv"
ruta_directorio = "../../Resultados/Resultados (etapa 1 y 2)/Análisis final"
os.makedirs(ruta_directorio, exist_ok=True)
ruta_csv = os.path.join(ruta_directorio, "correlaciones_completas_targets.csv")

print("Cargando dataset de entrenamiento...")
df_train = pd.read_csv(ruta_train)

print("Calculando matriz de correlación de Spearman (esto puede tomar unos segundos)...")
# Calculamos la correlación
corr_matrix = df_train.corr(method='spearman')

# Definir los targets
targets = ["MORTALIDAD", "SEVERIDAD", "CONSUMO_RECURSOS"]

# Extraer solo las filas de las variables predictoras (excluyendo los targets)
predictores = [col for col in corr_matrix.columns if col not in targets]
tabla_correlaciones = corr_matrix.loc[predictores, targets].copy()

# Calcular el poder predictivo general (Promedio del valor absoluto de las 3 correlaciones)
# Esto nos ayuda a ordenar la tabla desde las variables más fuertes a las más débiles
tabla_correlaciones['PODER_PREDICTIVO_PROMEDIO'] = tabla_correlaciones[targets].abs().mean(axis=1)

# Ordenar de mayor a menor poder predictivo
tabla_correlaciones = tabla_correlaciones.sort_values(by='PODER_PREDICTIVO_PROMEDIO', ascending=False)

# Redondear a 4 decimales para que el CSV quede impecable
tabla_correlaciones = tabla_correlaciones.round(4)

# Exportar a CSV (index_label le da un nombre a la primera columna)
tabla_correlaciones.to_csv(ruta_csv, index_label="VARIABLE_PREDICTORA")

print(f"✔️ ¡Tabla de correlaciones guardada exitosamente en:\n{ruta_csv}")
print("\n--- Vista previa del Top 5 de variables más influyentes ---")
print(tabla_correlaciones.head(5))

Cargando dataset de entrenamiento...
Calculando matriz de correlación de Spearman (esto puede tomar unos segundos)...
✔️ ¡Tabla de correlaciones guardada exitosamente en:
../../Resultados/Resultados (etapa 1 y 2)/Análisis final\correlaciones_completas_targets.csv

--- Vista previa del Top 5 de variables más influyentes ---
                                                MORTALIDAD  SEVERIDAD  \
NUM_COMORBILIDADES                                  0.2267     0.6140   
TIPO_ACTIVIDAD_ATENCION_AMBULATORIA                -0.0915    -0.6035   
TIPO_ACTIVIDAD_HOSPITALIZACION                      0.0915     0.6035   
TIPO_INGRESO_PROGRAMADO                            -0.2784    -0.4777   
TIPO_INGRESO_NO_PROGRAMADO_URGENCIAOBSTETRICIA      0.2784     0.4777   

                                                CONSUMO_RECURSOS  \
NUM_COMORBILIDADES                                        0.3925   
TIPO_ACTIVIDAD_ATENCION_AMBULATORIA                      -0.3155   
TIPO_ACTIVIDAD_HOSPITALIZACION  

In [38]:
import pandas as pd
import os
import numpy as np

def tabla_frecuencias(carpeta, archivos, columna):
    """
    Genera una tabla de frecuencias anuales para una columna categórica.
    Filas = categorías, Columnas = años.
    """
    resultados = {}
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        # Evitar warnings de tipos mezclados
        df = pd.read_csv(ruta, low_memory=False)
        año = archivo[-8:-4]
        # Convertir la columna a string para evitar conflictos de tipos
        resultados[año] = df[columna].astype(str).value_counts()

    # Combinar resultados en un DataFrame
    tabla = pd.DataFrame(resultados).fillna(0).astype(int)
    return tabla


def tablas_estadisticas(carpeta, archivos, columnas_numericas):
    """
    Genera una tabla de estadísticas anuales para cada columna numérica.
    Devuelve un diccionario {columna: DataFrame}.
    Cada DataFrame tiene filas = métricas (mean, var, min, max),
    columnas = años.
    """
    resultados = {col: {} for col in columnas_numericas}

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        df = pd.read_csv(ruta, low_memory=False)
        año = archivo[-8:-4]

        for col in columnas_numericas:
            if col in df.columns:
                serie = pd.to_numeric(df[col], errors="coerce")
                resultados[col][año] = {
                    "mean": round(serie.mean(), 2),
                    "var": round(serie.var(), 2),
                    "min": int(serie.min()) if not pd.isna(serie.min()) else np.nan,
                    "max": int(serie.max()) if not pd.isna(serie.max()) else np.nan
                }

    # Convertir cada diccionario en DataFrame
    tablas = {}
    for col, stats in resultados.items():
        tablas[col] = pd.DataFrame(stats)

    return tablas

# Ejemplo de uso con MORTALIDAD
carpeta = "../../Datos/Datos oncológicos (procesados)"
archivos = [
    "GRD_ONCOLOGIA_PROC_2019.csv",
    "GRD_ONCOLOGIA_PROC_2020.csv",
    "GRD_ONCOLOGIA_PROC_2021.csv",
    "GRD_ONCOLOGIA_PROC_2022.csv",
    "GRD_ONCOLOGIA_PROC_2023.csv",
    "GRD_ONCOLOGIA_PROC_2024.csv"
]

In [33]:
tabla_frecuencias(carpeta, archivos, "CATEGORIA_CANCER")

,2019,2020,2021,2022,2023,2024
CATEGORIA_CANCER,,,,,,
"C00-C14: Labio, cavidad oral y faringe",640,461,524,641,623,802
C15-C26: Órganos digestivos,12422,9973,10840,12339,14218,15161
C30-C39: Órganos respiratorios e intratorácicos,2250,1689,1854,2057,2566,2586
C40-C41: Hueso y cartílago articular,392,308,310,394,420,486
C43-C44: Melanoma y otras neoplasias malignas de piel,2224,1079,1286,1545,1575,1796
C45-C49: Tejidos mesoteliales y tejidos blandos,490,355,427,467,575,667
C50: Mama,6412,4707,5346,6916,7786,7539
C51-C58: Órganos genitales femeninos,3751,3115,3388,3652,4321,4788
C60-C63: Órganos genitales masculinos,2728,1866,1991,2483,2985,3067


In [39]:
tabla_frecuencias(carpeta, archivos, "MORTALIDAD")

,2019,2020,2021,2022,2023,2024
MORTALIDAD,,,,,,
0,42827,33184,36751,42205,48690,51555
1,2748,2004,2014,2273,2713,3018


In [4]:
tabla_frecuencias(carpeta, archivos, "SEVERIDAD")

,2019,2020,2021,2022,2023,2024
SEVERIDAD,,,,,,
0,5629,2759,4537,6290,8236,8803
1,17329,13090,13728,14538,14959,15587
2,13830,11448,12210,13702,16459,17006
3,8698,7873,8290,9948,11749,13177


In [5]:
tabla_frecuencias(carpeta, archivos, "CONSUMO_RECURSOS")

,2019,2020,2021,2022,2023,2024
CONSUMO_RECURSOS,,,,,,
0,15252,12023,12973,15313,17262,18355
1,15613,11709,12951,14407,17517,18395
2,14621,11438,12841,14758,16624,17823


In [6]:
tabla_frecuencias(carpeta, archivos, "SEXO")

,2019,2020,2021,2022,2023,2024
SEXO,,,,,,
MUJER,25596,19648,21709,25245,29186,30800
HOMBRE,19890,15522,17056,19233,22217,23773


In [7]:
tabla_frecuencias(carpeta, archivos, "ES_PUEBLO_ORIGINARIO")

,2019,2020,2021,2022,2023,2024
ES_PUEBLO_ORIGINARIO,,,,,,
0,44938,34566,38222,43822,50572,53557
1,548,604,543,656,831,1016


In [8]:
tabla_frecuencias(carpeta, archivos, "ES_EXTRANJERO")

,2019,2020,2021,2022,2023,2024
ES_EXTRANJERO,,,,,,
0,44785,34435,37713,43072,49496,52360
1,701,735,1052,1406,1907,2213


In [41]:
tabla_frecuencias(carpeta, archivos, "PREVISION")

,2019,2020,2021,2022,2023,2024
PREVISION,,,,,,
FONASA A,8085,6215,6151,6792,7226,6938
FONASA B,24750,19404,21637,25640,29886,31955
FONASA C,5140,3956,4725,5098,6297,7059
FONASA D,7137,5320,5990,6643,7722,8338
OTRAS PREVISIONES (FFAA/PRIVADO),374,275,262,305,272,283


In [10]:
tabla_frecuencias(carpeta, archivos, "SERVICIO_SALUD")

,2019,2020,2021,2022,2023,2024
SERVICIO_SALUD,,,,,,
REGION METROPOLITANA,15540,11451,13076,15149,17667,18559
MACROZONA SUR,12992,10821,12274,13481,15222,16487
MACROZONA CENTRO,10302,7413,7440,9170,10838,11567
MACROZONA NORTE,5397,4557,5105,5665,6586,6809
MACROZONA AUSTRAL,1255,928,870,1013,1090,1151


In [11]:
tabla_frecuencias(carpeta, archivos, "TIPO_PROCEDENCIA")

,2019,2020,2021,2022,2023,2024
TIPO_PROCEDENCIA,,,,,,
ATENCION PRIMARIA,381,348,282,290,283,356
ESPECIALIDAD,29710,21300,24997,28909,33224,35189
INSTITUCIONAL Y ALTERNATIVA,88,166,190,183,570,828
PROGRAMAS Y ELECTIVA,233,164,330,1485,1316,927
TRASLADO DE RED,3156,2873,2661,2789,3329,3686
URGENCIA,11918,10319,10305,10822,12681,13587


In [12]:
tabla_frecuencias(carpeta, archivos, "ESPECIALIDAD_MEDICA")

,2019,2020,2021,2022,2023,2024
ESPECIALIDAD_MEDICA,,,,,,
CIRUGIA ESPECIALIZADA,3380,2594,3211,3970,5032,5287
CIRUGIA GENERAL Y DIGESTIVA,16771,13717,15169,17222,19215,19885
GINECOLOGIA Y OBSTETRICIA,5426,4609,4972,5733,6719,7269
MEDICINA INTERNA Y CRITICA,6716,5411,5339,5954,7134,7850
ONCOLOGIA Y HEMATOLOGIA,4274,3048,3361,3763,4063,4565
OTRAS ESPECIALIDADES,3690,1674,2101,2390,2820,3027
PEDIATRIA,368,337,365,364,340,415
UROLOGIA,4861,3780,4247,5082,6080,6275


In [13]:
tabla_frecuencias(carpeta, archivos, "TIPO_ACTIVIDAD")

,2019,2020,2021,2022,2023,2024
TIPO_ACTIVIDAD,,,,,,
HOSPITALIZACION,39860,32411,34228,38188,43167,45770
ATENCION AMBULATORIA,5626,2759,4537,6290,8236,8803


In [14]:
tabla_frecuencias(carpeta, archivos, "SERVICIOINGRESO")

,2019,2020,2021,2022,2023,2024
SERVICIOINGRESO,,,,,,
CUIDADOS CRITICOS (PAC),1381,1448,1660,2303,3048,3496
HOSPITALIZACION MEDICO-QUIRURGICA,27186,21238,22563,26498,30090,32139
MATERNO-INFANTIL,5639,4803,5435,6098,6789,7103
OTRAS UNIDADES,5465,4214,4617,4258,4999,4629
TRANSITORIO Y PABELLON,3795,2782,4133,4965,6105,6882
URGENCIA,2020,685,357,356,372,324


In [15]:
tabla_frecuencias(carpeta, archivos, "TRASLADO_MACROZONA")

,2019,2020,2021,2022,2023,2024
TRASLADO_MACROZONA,,,,,,
0,43960,33904,37336,42914,49870,52781
1,1526,1266,1429,1564,1533,1792


In [16]:
tabla_frecuencias(carpeta, archivos, "TIPO_INGRESO")

,2019,2020,2021,2022,2023,2024
TIPO_INGRESO,,,,,,
PROGRAMADO,29828,21687,25187,30174,34718,36230
NO PROGRAMADO (URGENCIA/OBSTETRICIA),15658,13483,13578,14304,16685,18343


In [17]:
tabla_frecuencias(carpeta, archivos, "COMORBILIDAD_PRINCIPAL")

,2019,2020,2021,2022,2023,2024
COMORBILIDAD_PRINCIPAL,,,,,,
CODIGOS PROVISIONALES (COVID),20,264,241,503,232,131
ENDOCRINAS Y METABOLICAS,5070,3750,4739,5609,6544,7060
INFECCIOSAS Y PARASITARIAS,699,633,684,747,916,977
MATERNO-INFANTILES Y CONGENITAS,126,110,109,152,152,138
OJO Y OIDO,306,187,240,276,377,356
PIEL Y TEJIDO SUBCUTANEO,324,234,255,260,349,458
SANGRE E INMUNIDAD,3284,2717,2668,2833,3329,3905
SINTOMAS Y HALLAZGOS,2005,1680,1863,1960,2409,2734
SIN_COMORBILIDAD,10494,7415,7964,9067,9672,9015


In [18]:
tabla_frecuencias(carpeta, archivos, "SERVICIOINGRESO")

,2019,2020,2021,2022,2023,2024
SERVICIOINGRESO,,,,,,
CUIDADOS CRITICOS (PAC),1381,1448,1660,2303,3048,3496
HOSPITALIZACION MEDICO-QUIRURGICA,27186,21238,22563,26498,30090,32139
MATERNO-INFANTIL,5639,4803,5435,6098,6789,7103
OTRAS UNIDADES,5465,4214,4617,4258,4999,4629
TRANSITORIO Y PABELLON,3795,2782,4133,4965,6105,6882
URGENCIA,2020,685,357,356,372,324


In [42]:
tabla_frecuencias(carpeta, archivos, "CATEGORIA_CANCER")

,2019,2020,2021,2022,2023,2024
CATEGORIA_CANCER,,,,,,
"C00-C14: Labio, cavidad oral y faringe",640,461,524,641,623,802
C15-C26: Órganos digestivos,12399,9967,10840,12339,14218,15161
C30-C39: Órganos respiratorios e intratorácicos,2248,1689,1854,2057,2566,2586
C40-C41: Hueso y cartílago articular,391,308,310,394,420,486
C43-C44: Melanoma y otras neoplasias malignas de piel,2221,1079,1286,1545,1575,1796
C45-C49: Tejidos mesoteliales y tejidos blandos,490,355,427,467,575,667
C50: Mama,6381,4704,5346,6916,7786,7539
C51-C58: Órganos genitales femeninos,3744,3115,3388,3652,4321,4788
C60-C63: Órganos genitales masculinos,2725,1864,1991,2483,2985,3067


In [27]:
columnas_numericas = ["EDAD", "CARGA_ONCOLOGICA", "NUM_COMORBILIDADES"]

tablas = tablas_estadisticas(carpeta, archivos, columnas_numericas)

tablas["EDAD"]

,2019,2020,2021,2022,2023,2024
mean,58.72,57.94,58.07,58.99,59.54,59.93
var,369.76,378.83,359.20,331.98,323.76,327.03
min,0.00,0.00,0.00,0.00,0.00,0.00
max,118.00,102.00,110.00,102.00,101.00,104.00


In [28]:
tablas["CARGA_ONCOLOGICA"]

,2019,2020,2021,2022,2023,2024
mean,1.31,1.32,1.32,1.32,1.33,1.35
var,0.46,0.47,0.46,0.48,0.52,0.54
min,1.00,1.00,1.00,1.00,1.00,1.00
max,8.00,8.00,10.00,10.00,8.00,10.00


In [29]:
tablas["NUM_COMORBILIDADES"]

,2019,2020,2021,2022,2023,2024
mean,2.68,2.95,3.08,3.23,3.47,3.74
var,8.45,10.01,11.08,12.22,13.40,14.82
min,0.00,0.00,0.00,0.00,0.00,0.00
max,32.00,32.00,32.00,32.00,32.00,33.00
